# Outlier Analysis
Goal: identify which time buckets are statistical outliers, whether outliers co-occur across features, and whether they cluster temporally.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/preprocess/features.csv', index_col=0)

# Drop the ID column — not a feature
feature_cols = [c for c in df.columns if c != 'time_bucket_id']

print(df.shape)
df.head()

In [ ]:
# Temporal train/test split — must split before computing any statistics
# Sort by time to ensure no future data leaks into training bounds
df = df.sort_values('time_bucket_id').reset_index(drop=True)

train_size = int(len(df) * 0.8)
train_df = df.iloc[:train_size].copy()
test_df  = df.iloc[train_size:].copy()

print(f"Train: {len(train_df)} buckets (bucket {train_df['time_bucket_id'].min()} – {train_df['time_bucket_id'].max()})")
print(f"Test:  {len(test_df)} buckets  (bucket {test_df['time_bucket_id'].min()} – {test_df['time_bucket_id'].max()})")

# Compute IQR bounds from TRAINING data only, then apply to full dataset
# This prevents test distribution from influencing what counts as an outlier

def compute_iqr_bounds(train_series, k=3.0):
    q1, q3 = train_series.quantile(0.25), train_series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

# Fit bounds on train, store them for later use in encoding
iqr_bounds = {
    col: compute_iqr_bounds(train_df[col])
    for col in feature_cols
}

def apply_outlier_mask(series, col):
    lower, upper = iqr_bounds[col]
    return (series < lower) | (series > upper)

# Apply train-fit bounds to the full dataset for EDA visibility
outlier_matrix = pd.DataFrame(
    {col: apply_outlier_mask(df[col], col) for col in feature_cols},
    index=df.index
)
outlier_matrix['outlier_count'] = outlier_matrix[feature_cols].sum(axis=1)

# Also build train/test specific matrices for later
train_outlier_matrix = outlier_matrix.iloc[:train_size]
test_outlier_matrix  = outlier_matrix.iloc[train_size:]

print(f"Buckets with ≥1 outlier feature:  {(outlier_matrix['outlier_count'] >= 1).sum()}")
print(f"Buckets with ≥3 outlier features: {(outlier_matrix['outlier_count'] >= 3).sum()}")
print(f"Buckets with ≥5 outlier features: {(outlier_matrix['outlier_count'] >= 5).sum()}")

In [ ]:
def iqr_outlier_mask(series, k=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - k * iqr) | (series > q3 + k * iqr)

outlier_matrix = pd.DataFrame(
    {col: iqr_outlier_mask(df[col]) for col in feature_cols},
    index=df.index
)

# How many buckets are outliers in at least one feature?
outlier_matrix['outlier_count'] = outlier_matrix.sum(axis=1)
print(f"Buckets with ≥1 outlier feature:  {(outlier_matrix['outlier_count'] >= 1).sum()}")
print(f"Buckets with ≥3 outlier features: {(outlier_matrix['outlier_count'] >= 3).sum()}")
print(f"Buckets with ≥5 outlier features: {(outlier_matrix['outlier_count'] >= 5).sum()}")

## 2. Which features flag the most buckets as outliers?

In [ ]:
feature_outlier_counts = outlier_matrix[feature_cols].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(14, 4))
feature_outlier_counts.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Number of outlier buckets per feature (IQR × 3)')
ax.set_ylabel('Bucket count')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Features that never flag anything are low-signal — note them
zero_signal = feature_outlier_counts[feature_outlier_counts == 0].index.tolist()
print(f"Features with zero outliers (consider dropping): {zero_signal}")

## 3. Suspicious buckets: co-occurring outliers across features
A bucket that is extreme in many features simultaneously is a stronger anomaly candidate than one that spikes in only one feature.

In [ ]:
suspicious = (
    outlier_matrix[outlier_matrix['outlier_count'] >= 3]
    .sort_values('outlier_count', ascending=False)
)

# Show which features each suspicious bucket is extreme in
for idx in suspicious.index:
    bucket_id = df.loc[idx, 'time_bucket_id']
    n = suspicious.loc[idx, 'outlier_count']
    flagged = [c for c in feature_cols if outlier_matrix.loc[idx, c]]
    print(f"Row {idx} | bucket {bucket_id:>3} | {n} outlier features: {flagged}")

## 4. Temporal clustering: are outlier buckets consecutive?
Consecutive outlier buckets suggest a sustained event (attack campaign, scan). Isolated buckets suggest one-off probes.

In [ ]:
df_plot = df[['time_bucket_id']].copy()
df_plot['outlier_count'] = outlier_matrix['outlier_count'].values
df_plot['is_suspicious'] = (df_plot['outlier_count'] >= 3).astype(int)

fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

axes[0].bar(df_plot['time_bucket_id'], df_plot['outlier_count'], color='steelblue', width=0.8)
axes[0].axhline(3, color='red', linestyle='--', linewidth=1, label='threshold (3 features)')
axes[0].set_ylabel('# outlier features')
axes[0].set_title('Outlier feature count per time bucket')
axes[0].legend()

axes[1].bar(df_plot['time_bucket_id'], df_plot['is_suspicious'], color='crimson', width=0.8)
axes[1].set_ylabel('suspicious (0/1)')
axes[1].set_xlabel('Time bucket (day)')
axes[1].set_title('Suspicious buckets (≥3 outlier features)')

plt.tight_layout()
plt.show()

## 5. Outlier co-occurrence heatmap
Which pairs of features tend to be outliers in the same buckets? High co-occurrence means the features are capturing the same event — one of them may be redundant.

In [ ]:
# Only include features that have at least 1 outlier bucket
active_features = [c for c in feature_cols if outlier_matrix[c].sum() > 0]
cooccurrence = outlier_matrix[active_features].T.dot(outlier_matrix[active_features]).astype(int)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cooccurrence, annot=True, fmt='d', cmap='Blues',
    linewidths=0.3, ax=ax, cbar_kws={'label': 'shared outlier buckets'}
)
ax.set_title('Outlier co-occurrence across features\n(diagonal = total outlier buckets for that feature)')
plt.tight_layout()
plt.show()

## 6. Outlier vs. non-outlier bucket comparison
For each feature, compare the median value in suspicious buckets vs. normal buckets.

In [ ]:
df_compare = df[feature_cols].copy()
df_compare['suspicious'] = (outlier_matrix['outlier_count'].values >= 3)

summary = df_compare.groupby('suspicious')[feature_cols].median().T
summary.columns = ['normal', 'suspicious']
summary['ratio'] = (summary['suspicious'] / summary['normal'].replace(0, np.nan)).round(2)
summary = summary.sort_values('ratio', ascending=False)

print("Feature medians: suspicious vs. normal buckets")
print(summary.to_string())

## 7. Decision: should you encode outlier flags as features?

Based on the analysis above:
- If suspicious buckets show **consistent, large ratios** across multiple features → the raw scaled values already carry strong signal; flags are optional
- If a feature has **1–2 extreme outlier buckets** that dwarf everything else → a binary flag may help tree models; log-transform + flag is best
- If suspicious buckets are **temporally clustered** → consider adding a `rolling_outlier_count` (rolling window of outlier_count) as a feature to give models temporal context

Run the cell below to add optional outlier flag columns to the feature set.

In [ ]:
# Optional: add per-feature binary outlier flags and a rolling outlier count
# Only add flags for features where the outlier buckets show large ratio vs normal (adjust threshold as needed)

HIGH_RATIO_THRESHOLD = 5.0  # suspicious median is 5× the normal median

flag_features = summary[summary['ratio'] >= HIGH_RATIO_THRESHOLD].index.tolist()
print(f"Adding outlier flags for: {flag_features}")

df_enriched = df.copy()

for col in flag_features:
    df_enriched[f'{col}_outlier'] = outlier_matrix[col].astype(int)

# Rolling outlier count (window=3 days) — gives model temporal context
df_enriched = df_enriched.sort_values('time_bucket_id').reset_index(drop=True)
df_enriched['rolling_outlier_count'] = (
    outlier_matrix['outlier_count']
    .reindex(df_enriched.index)
    .rolling(window=3, min_periods=1)
    .sum()
    .values
)

print(f"\nFinal shape: {df_enriched.shape}")
df_enriched.head()

In [ ]:
# Save enriched features
df_enriched.to_csv('../data/preprocess/features_enriched.csv', index=False)
print('Saved to data/preprocess/features_enriched.csv')